# Credit Scoring · Exploratory Data Analysis

## Summary

This notebook profiles labeled Home Credit applications and audits the
depth-zero feature tables before model development.

## Context and methods

The unit of analysis is a credit case. The binary target records whether the
loan defaulted under the dataset's definition. The analytical population
consists of the labeled cases provided by Home Credit.

We use decision weeks for chronological evaluation and preserve entire weeks
at split boundaries. The final window is reserved for future model evaluation.
EDA involving outcomes is restricted to the development window. Before training,
confirm the label observation horizon and allow any required gap between windows.

[Source and feature definitions](https://www.kaggle.com/competitions/home-credit-credit-risk-model-stability/data)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_dir = project_root / "data" / "raw"
train_dir = raw_dir / "parquet_files" / "train"
base_path = train_dir / "train_base.parquet"

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True,
                     "grid.alpha": 0.2, "axes.spines.top": False,
                     "axes.spines.right": False})

## Data

### Validate the base table

In [ ]:
if not base_path.is_file():
    raise FileNotFoundError(f"Expected {base_path}. Follow the download steps in README.md.")

base = pd.read_parquet(base_path, columns=["case_id", "date_decision", "WEEK_NUM", "target"])
base["date_decision"] = pd.to_datetime(base["date_decision"], errors="raise")
assert base.notna().all().all()
assert base["case_id"].is_unique
assert set(base["target"].unique()) == {0, 1}

weeks = sorted(base["WEEK_NUM"].unique())
assert len(weeks) >= 10
calibration_start = weeks[int(0.60 * len(weeks))]
evaluation_start = weeks[int(0.80 * len(weeks))]
development = base.loc[base["WEEK_NUM"] < calibration_start].copy()

window = pd.Series("Development", index=base.index)
window.loc[base["WEEK_NUM"] >= calibration_start] = "Calibration and policy"
window.loc[base["WEEK_NUM"] >= evaluation_start] = "Final evaluation"
split_summary = base.groupby(window, sort=False).agg(
    cases=("case_id", "size"), weeks=("WEEK_NUM", "nunique"),
    first_decision=("date_decision", "min"), last_decision=("date_decision", "max"),
).sort_values("first_decision")
assert (split_summary["last_decision"].iloc[:-1].to_numpy()
        < split_summary["first_decision"].iloc[1:].to_numpy()).all()
display(split_summary)

## Results

### Weekly volume and observed default rate in development

In [ ]:
weekly = development.groupby("WEEK_NUM").agg(
    cases=("case_id", "size"), default_rate=("target", "mean"),
)
print(f"Development cases: {len(development):,}")
print(f"Development default rate: {development['target'].mean():.2%}")

figure, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 6), layout="constrained")
weekly["cases"].plot(ax=axes[0], color="#235789", title="Weekly volume in development")
weekly["default_rate"].plot(ax=axes[1], color="#a44200", title="Observed weekly default rate")
axes[0].set_ylabel("Cases")
axes[1].set_ylabel("Share of cases")
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
axes[1].set_xlabel("Decision week")
plt.show()

### Audit static table coverage and feature missingness

Each static table is expected to contain at most one row per case across its
shards. Coverage is measured against development cases. Missingness includes
cases with no matching row, as it would after a left join. Reading columns
in batches limits peak memory use.

In [ ]:
static_tables = {
    "static_0": sorted(train_dir.glob("train_static_0_*.parquet")),
    "static_cb_0": sorted(train_dir.glob("train_static_cb_0*.parquet")),
}
development_ids = pd.Index(development["case_id"])
coverage_rows = []
missingness_rows = []

for table_name, paths in static_tables.items():
    if not paths:
        raise FileNotFoundError(f"No parquet files found for {table_name} in {train_dir}")
    shards = [pq.ParquetFile(path) for path in paths]
    features = [name for name in shards[0].schema_arrow.names if name != "case_id"]
    assert all(set(shard.schema_arrow.names) == set(features) | {"case_id"} for shard in shards)
    all_ids = pd.Index(pd.concat([
        shard.read(columns=["case_id"]).to_pandas()["case_id"] for shard in shards
    ], ignore_index=True))
    assert not all_ids.hasnans and all_ids.is_unique, table_name

    matched = development_ids.isin(all_ids).sum()
    coverage_rows.append({
        "table": table_name, "features": len(features), "matched cases": matched,
        "coverage": matched / len(development),
        "cases outside base": (~all_ids.isin(base["case_id"])).sum(),
    })

    for start in range(0, len(features), 24):
        columns = features[start:start + 24]
        observed = pd.Series(0, index=columns, dtype="int64")
        for shard in shards:
            for batch in shard.iter_batches(columns=["case_id", *columns]):
                frame = batch.to_pandas()
                mask = frame["case_id"].isin(development_ids)
                observed += frame.loc[mask, columns].count()
        missingness_rows.extend(
            {"table": table_name, "feature": name, "missing share": 1 - count / len(development)}
            for name, count in observed.items()
        )

coverage = pd.DataFrame(coverage_rows).set_index("table")
missingness = pd.DataFrame(missingness_rows)
display(coverage.style.format({"coverage": "{:.1%}"}))

### Inspect the features with the highest missingness

In [ ]:
top_missing = missingness.nlargest(12, "missing share").sort_values("missing share")
labels = top_missing["table"] + " / " + top_missing["feature"]
figure, axis = plt.subplots(figsize=(10, 5), layout="constrained")
axis.barh(labels, top_missing["missing share"], color="#235789")
axis.xaxis.set_major_formatter(PercentFormatter(1))
axis.set(xlim=(0, 1), xlabel="Share of development cases with a missing value",
         title="Highest missingness across depth-zero features")
plt.show()

display(missingness.groupby("table")["missing share"].agg(
    median="median", fully_missing=lambda values: values.eq(1).sum()
).style.format({"median": "{:.1%}"}))

## Takeaways

Use the coverage audit to preserve the base population when joining features.
Review feature definitions and availability at decision time before modeling.
Track ranking and probability calibration across chronological windows. Use
the calibration and policy window to choose a threshold, then evaluate it once
on the final window.